# Assessed value per acre

A parcel map with a computed median and explicit coverage notes.

This is an editable worked example of [`assessed_value_per_acre.py`](../src/graphics/assessed_value_per_acre.py). It runs Python SDK calls directly—no website, CLI subprocess, or registered build dispatcher. The canonical definition remains the publishing source of truth.

**What the numbers mean:** Assessed land plus improvements divided by recorded area is not sale price, tax bill, or land-only value. Recorded zero and missing records remain distinct.

Start with **Run All**, inspect the data table and preview, then change the title or a visual encoding in step 4. Parcel examples load the full city and can take several minutes.


## 1. Open the libraries

Use the environment in [README.md](README.md). Paths below locate the checkout, not a personal machine.


In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder or anywhere inside the Detroit checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "strongtowns-data.lock.json").is_file()
             and (p / "projects/graphics/src/graphics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the strongtowns-detroit checkout; see README.md.")
sys.path.insert(0, str(ROOT / "src"))
GRAPHICS = ROOT / "projects/graphics"
FORUM = ROOT / "projects/detroit-land-use-forum"
import strongtowns_graphics as graphics_sdk
if not hasattr(graphics_sdk, "GraphicInput"):
    raise RuntimeError(
        "This kernel has an older graphics SDK. Restart Jupyter with the uv command "
        "in README.md so it uses this project's locked dependencies."
    )
from IPython.display import SVG, display
from strongtowns_graphics import (
    GraphicFormat, GRAPHIC_FORMAT_SPECS, render_graphic_canvas,
    render_graphic_svg, write_graphic_bundle,
)

NOTEBOOK_NAME = 'assessed_value_per_acre'


In [ ]:
import sys

from pathlib import Path

import geopandas as gpd

sys.path.insert(0, str(FORUM / "assessed-value-per-acre"))

from build_assessed_value_asset import (  # noqa: E402
    build_graphic,
    classify,
    concentration,
)

from strongtowns_graphics import (
    GraphicInput,
    MobileMapInset,
    MobileMapPocket,
    graphic_definition,
    map_on_mobile,
)


## 2. Resolve the prepared data

The aliases below name the inputs you will read. The data SDK resolves only the snapshots pinned by this project. Change prepared inputs through a reviewed data lock update, not by pointing at a moving latest file.


In [ ]:
from strongtowns_data import DataBuildSystem, DataLock, DataRepository
from strongtowns_detroit.repositories import data_repository
from strongtowns_graphics import GraphicBuildContext

requirements = (
        GraphicInput("parcels", "detroit.parcels.raw", "raw.geojson"),
        GraphicInput("roads", "detroit.base-units.streets.raw", "raw.geojson"),
    )
lock = DataLock.load(ROOT / "strongtowns-data.lock.json")
repository = DataRepository(DataBuildSystem.find(data_repository()))
paths, provenance = {}, {}
for requirement in requirements:
    try:
        reference = lock.asset(requirement.dataset_id)
        paths[requirement.alias] = repository.artifact(reference, requirement.artifact)
        if not paths[requirement.alias].is_file():
            raise FileNotFoundError(paths[requirement.alias])
        provenance[requirement.alias] = {
            "dataset": requirement.dataset_id,
            "artifact": requirement.artifact,
            "snapshot": reference.snapshot_id,
            "manifest_sha256": reference.manifest_sha256,
        }
    except (ValueError, KeyError, FileNotFoundError) as error:
        raise RuntimeError(
            f"Prepared input unavailable: {requirement.dataset_id}/{requirement.artifact}. "
            "Ask the data maintainer to restore the pinned snapshot in strongtowns-data; "
            "this notebook never fetches data or changes the lock."
        ) from error
context = GraphicBuildContext(paths)
provenance


## 3. Prepare and inspect the table

This follows the existing graphic’s data selection and calculations. The displayed rows are a preview; the graphic uses the full prepared table.


In [ ]:
frame = gpd.read_file(
    context.input("parcels"),
    columns=["parcel_id", "amt_assessed_value", "total_square_footage", "geometry"],
).rename(columns={"amt_assessed_value": "assessed_value"})

frame = classify(frame)

recorded = frame[frame["recorded"]]

zero = int(recorded["assessed"].eq(0).sum())

unknown = int((~frame["recorded"]).sum())

share = concentration(frame)


In [ ]:
display(frame.drop(columns="geometry").head(8))


## 4. Build the graphic with the SDK

Edit `title`, `subtitle`, colors, legends, or explicit encodings here. Keep sources and descriptions accurate when changing data. The parcel and travel examples reuse existing project map-drawing helpers, then call SDK composition functions; the bar, point-map, and continuous-choropleth examples expose their renderer calls directly.


In [ ]:
graphic = build_graphic(
    frame,
    roads_path=context.input("roads"),
    title="Detroit's assessed property value per acre",
    subtitle=(
        "Total assessed land and improvement value divided by recorded "
        "parcel area"
    ),
    sources=(
        f"Coverage: {len(recorded):,} parcels with recorded area and "
        f"assessment; {zero:,} have a recorded assessment of $0; "
        f"{unknown:,} lack a usable area or assessment.",
        "Source: City of Detroit parcel assessment data downloaded in 2026.",
        "Values are nominal assessor records and have not been adjusted "
        "for exemptions or assessment-year differences.",
    ),
    description=(
        "Parcel map comparing total assessed land and improvement value "
        f"per acre. {share:.0%} of recorded assessed value is concentrated "
        "on 10 percent of recorded parcel acreage."
    ),
)

median = float(graphic.metadata["median_assessed_value_per_acre"])

graphic = map_on_mobile(
    graphic,
    map_height=800,
    insets=(
        MobileMapInset.hero_statistic(
            pocket=MobileMapPocket.LOWER_RIGHT,
            value=f"${median / 1000:.0f}K",
            label=(
                "MEDIAN ASSESSED VALUE",
                "PER ACRE ACROSS PARCELS",
                "WITH USABLE RECORDS",
            ),
        ),
    ),
)

graphics = {"assessed-value-per-acre": graphic}


## 5. Choose the Instagram format and preview

Feed uses 1080 × 1350; story uses 1080 × 1920 with the library’s established content padding. Changing this enum preserves the publishing policy.


In [ ]:
# Change to GraphicFormat.INSTAGRAM_STORY for a story-sized export.
TARGET = GraphicFormat.INSTAGRAM_POST
EXPORT_PNG = True  # SVG and HTML work without the rsvg-convert system tool.
target = GRAPHIC_FORMAT_SPECS[TARGET]


In [ ]:
# Preview exactly the composition used by the export below.
for name, graphic in graphics.items():
    print(name)
    svg = (render_graphic_canvas(
        graphic, canvas_aspect_ratio=target.aspect_ratio,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding,
    ) if target.content_aspect_ratio else render_graphic_svg(
        graphic, aspect_ratio=target.aspect_ratio,
    ))
    display(SVG(svg))
    print("Alt text:", graphic.description or graphic.title_text)


## 6. Export

PNG is ready for Instagram; SVG and HTML retain the composition for inspection. Copy the adjacent alt-text file when posting. Exports go to the ignored `projects/graphics/output/notebooks/` directory and can be regenerated. Inspect all pages before sharing.


In [ ]:
import json
import shutil

output_dir = GRAPHICS / "output/notebooks" / NOTEBOOK_NAME / TARGET.value
formats = ("html", "svg", "png") if EXPORT_PNG else ("html", "svg")
if EXPORT_PNG and shutil.which("rsvg-convert") is None:
    raise RuntimeError(
        "PNG export needs rsvg-convert (see README.md). "
        "Set EXPORT_PNG = False above and rerun the export to save SVG/HTML now."
    )
for name, graphic in graphics.items():
    files = write_graphic_bundle(
        output_dir, name, graphic,
        aspect_ratio=target.aspect_ratio, png_width=target.png_width,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding, formats=formats,
    )
    (output_dir / f"{name}.alt.txt").write_text(
        graphic.description or graphic.title_text, encoding="utf-8"
    )
    for kind, path in files.items():
        print(f"{kind}: {path}")
# Keep the exact source identities alongside your exported graphics.
(output_dir / "sources.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)


## Try the pattern on another question

Make a copy of this notebook. Start by changing editorial wording, then inspect the explicit input table before changing a field or grouping. Keep units, unknown records, source coverage, and denominators visible. Changing geographic scope or a legal threshold requires reviewing the method and claim, not just replacing the title.

Use the other notebooks to compare stacked bars, categorized points, continuous parcel maps, and mobile map compositions. Clear outputs before committing a notebook; put publishing changes back into the canonical definition.
